# Setup and PEMSD4 reproduction

**What this notebook does**
1. Sets up a shared `CS637/` folder on Google Drive with the authors' code and the PEMSD4 data.
2. **Smoke test:** evaluates one shipped checkpoint and checks it against the authors' own training log.
3. Evaluates all six PEMSD4 checkpoints (Quantile and MAEMIS trios). No training.
4. Computes 15, 30 and 60 minute metrics plus interval coverage, and compares them with Table 1 of the paper.
5. Saves validation and test predictions as `.npz`.

## 0. Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, re, subprocess
BASE   = '/content/drive/MyDrive/CS637'          # shared team folder
AUTH   = f'{BASE}/uq4phy_traffic_network'        # authors' repo (never edited)
RAW    = f'{BASE}/data/raw'
OUT4   = f'{BASE}/outputs/pemsd4'                # prediction files for Person 3
P1     = f'{BASE}/p1'                            # our scripts and results (goes to reproduction/ in Git later)
DATA4  = '/content/pemsd4'                       # processed PEMSD4 on Colab's fast local disk (rebuilt each session)
for d in [RAW, OUT4, f'{P1}/scripts', f'{P1}/results', f'{P1}/logs']:
    os.makedirs(d, exist_ok=True)

MODELS = {   # short name -> authors' folder
    'Quantile':          'quantile_model_PEMSD4',
    'GPDE+Quantile':     'gpde_quantile_model_PEMSD4',
    'GPDE+Quantile+Phy': 'gpde_quantile_phy_model_PEMSD4',
    'MAEMIS':            'maemis_model_PEMSD4',
    'GPDE+MAEMIS':       'gpde_maemis_model_PEMSD4',
    'GPDE+MAEMIS+Phy':   'gpde_maemis_phy_model_PEMSD4_0.005',
}
PEMSD8_TRIO = ['quantile_model_PEMSD8', 'gpde_quantile_model_PEMSD8', 'gpde_quantile_phy_model_PEMSD8']  # for notebook 2
print('BASE =', BASE)

Mounted at /content/drive
BASE = /content/drive/MyDrive/CS637


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, numpy, scipy, yaml
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__,
      '| scipy', scipy.__version__, '| pyyaml', yaml.__version__, '| python', sys.version.split()[0])
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

name, memory.total [MiB]
Tesla T4, 15360 MiB
torch 2.11.0+cu130 | cuda True | numpy 2.1.3 | scipy 1.16.3 | pyyaml 6.0.3 | python 3.13.15


## 1. Authors' code

The full repo is about 1.2 GB with ~100 variant folders, so we clone **only the folders we need**: the six PEMSD4 variants and the PEMSD8 trio for notebook 2.
To add folders later (for Persons 2 and 3), run `git sparse-checkout add /<folder>/` inside `AUTH`.

In [ ]:
folders = list(MODELS.values()) + PEMSD8_TRIO
if not os.path.exists(f'{AUTH}/.git'):
    !git clone --filter=blob:none --no-checkout https://github.com/tianshubao1/uq4phy_traffic_network.git "{AUTH}"
    !git -C "{AUTH}" sparse-checkout set --no-cone {' '.join('/'+f+'/' for f in folders)}
    !git -C "{AUTH}" checkout
else:
    print('already cloned')
!git -C "{AUTH}" log -1 --format="authors' repo at commit %h (%cs)"
for f in folders:
    ckpts = sorted(glob.glob(f'{AUTH}/{f}/models/epo*.tar'))
    print(f'{f:40s} {len(ckpts):2d} checkpoints')

Cloning into '/content/drive/MyDrive/CS637/uq4phy_traffic_network'...
remote: Enumerating objects: 1568, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 1568 (delta 47), reused 0 (delta 0), pack-reused 1392 (from 1)
Receiving objects: 100% (1568/1568), 304.35 KiB | 1.46 MiB/s, done.
Resolving deltas: 100% (474/474), done.
Updating index flags: 100% (4544/4544), done.
remote: Enumerating objects: 201, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 201 (delta 79), reused 25 (delta 25), pack-reused 96 (from 1)
Receiving objects: 100% (201/201), 8.03 MiB | 8.66 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Updating files: 100% (372/372), done.
Your branch is up to date with 'origin/main'.
authors' repo at commit 7d41f27 (2026-05-24)
quantile_model_PEMSD4                     9 checkpoints
gpde_quantile_model_PEMSD4               13 checkpoints
gpd

## 2. PEMSD4 data

PEMS04 comes from the ASTGCN repository (`Davidham3/ASTGCN`, `data/PEMS04/pems04.npz`, shape 16992 × 307 × 3).

Notes for the README and critique:
- PEMS04 stores channels as **[flow, occupancy, speed]**. The authors' code expects **[speed, occupancy, flow]** (speed is the target and occupancy feeds the physics loss), so we reorder them.
- The processing script shipped in the PEMSD4 folders is the PEMSD8 script: it loads `pems8ndy.npy`, with 170 sensors and 17,856 steps hard-coded. Our `process_pems.py` follows the same logic (70/10/20 split, 12-in/12-out windows) with sizes read from the data.
- Their loop writes `x[i:] = …` instead of `x[i] = …`. The final arrays are the same, but it is very slow. Ours is vectorised.

In [ ]:
raw4 = f'{RAW}/pems04.npz'
if not os.path.exists(raw4):
    !wget -q -O "{raw4}" https://github.com/Davidham3/ASTGCN/raw/master/data/PEMS04/pems04.npz
import numpy as np
d = np.load(raw4)['data']
print('pems04.npz', d.shape, '| bytes', os.path.getsize(raw4))
assert d.shape == (16992, 307, 3), 'unexpected PEMS04 file'
print('channel means (raw order flow, occupancy, speed):', d.reshape(-1, 3).mean(0).round(3))
del d

pems04.npz (16992, 307, 3) | bytes 32956284
channel means (raw order flow, occupancy, speed): [2.11701e+02 5.30000e-02 6.34710e+01]


**Helper scripts.** The next cells write our scripts into `CS637/p1/scripts/`. They are tracked by us, and the authors' repo is never edited.

In [ ]:
%%writefile {P1}/scripts/process_pems.py
"""Build train/val/test.npz for the authors' DCRNN code from a raw PEMS array.

Faithful to the authors' 'PEMSD8 data processing.py' (70/10/20 split by time,
12-step input window, the next 12 steps as target, channels [speed, occupancy, flow]),
but vectorised (their loop writes x[i:] instead of x[i], which is slow but gives
the same final arrays) and with sizes taken from the data instead of hard-coded.
"""
import argparse, os, numpy as np
from numpy.lib.stride_tricks import sliding_window_view

def windows(seg, T=12):
    n = len(seg) - 2 * T + 1          # authors: len - 23 samples
    w = sliding_window_view(seg, 2 * T, axis=0)      # (n, nodes, ch, 24)
    w = np.moveaxis(w, -1, 1)[:n]                    # (n, 24, nodes, ch)
    return np.ascontiguousarray(w[:, :T]), np.ascontiguousarray(w[:, T:])

def main(a):
    raw = np.load(a.raw)
    data = raw[a.key] if hasattr(raw, 'files') else raw
    data = data[..., [int(c) for c in a.order.split(',')]].astype(np.float64)
    N = len(data); tr, va = int(N * 0.7), int(N * 0.8)
    os.makedirs(a.out, exist_ok=True)
    for name, seg in [('train', data[:tr]), ('val', data[tr:va]), ('test', data[va:])]:
        x, y = windows(seg)
        np.savez(os.path.join(a.out, name + '.npz'), x=x, y=y)
        print(f'{name}: x {x.shape} y {y.shape}')
    print('channel means [speed, occupancy, flow]:', data.reshape(-1, 3).mean(0).round(3))

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--raw', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--key', default='data')
    p.add_argument('--order', default='2,1,0', help='raw channel indices giving [speed, occupancy, flow]; PEMS04.npz is [flow, occupancy, speed]')
    main(p.parse_args())

Writing /content/drive/MyDrive/CS637/p1/scripts/process_pems.py


In [ ]:
%%writefile {P1}/scripts/eval_checkpoint.py
"""Evaluate one of the authors' shipped checkpoints and save val + test predictions.

Run from INSIDE a variant folder of the authors' repo, e.g.
    cd .../uq4phy_traffic_network/gpde_quantile_phy_model_PEMSD4
    python /path/eval_checkpoint.py --data_dir /content/pemsd4 --out /path/model.npz
The authors' code is not modified: we load their pretrained config with yaml.safe_load
(their run_demo uses yaml.load(f), which fails on PyYAML 6) and let torch.load read
their full checkpoint dict (PyTorch >= 2.6 defaults to weights_only=True).
"""
import argparse, glob, json, os, sys, time
import numpy as np, torch, yaml

_torch_load = torch.load
def _load(*a, **k):
    k.setdefault('weights_only', False)
    return _torch_load(*a, **k)
torch.load = _load

sys.path.insert(0, os.getcwd())
from lib.utils import load_graph_data                      # noqa: E402  (authors' code)
from model.pytorch.dcrnn_supervisor import DCRNNSupervisor  # noqa: E402

p = argparse.ArgumentParser()
p.add_argument('--data_dir', required=True)
p.add_argument('--out', required=True)
p.add_argument('--log_dir', default='/content/p1_eval_logs')
p.add_argument('--epoch', type=int, default=None, help='default: epoch in the pretrained config')
a = p.parse_args()

cfg_path = sorted(glob.glob('data/model/pretrained/*/config.yaml'))[0]
cfg = yaml.safe_load(open(cfg_path))
saved = sorted(int(f[3:-4]) for f in os.listdir('models') if f.startswith('epo') and f.endswith('.tar'))
epoch = a.epoch if a.epoch is not None else cfg['train']['epoch']
assert epoch in saved, f'epo{epoch}.tar not in models/ (have {saved})'
if epoch != max(saved):
    print(f'NOTE: config epoch {epoch} is not the latest saved checkpoint ({max(saved)})')
cfg['data']['dataset_dir'] = a.data_dir
cfg['train']['epoch'] = epoch
cfg['train']['log_dir'] = os.path.join(a.log_dir, os.path.basename(os.getcwd()))

_, _, adj = load_graph_data(cfg['data']['graph_pkl_filename'])
t0 = time.time()
sup = DCRNNSupervisor(adj_mx=adj, **cfg)
result, meta = {}, {'variant': os.path.basename(os.getcwd()), 'epoch': epoch, 'config': cfg_path,
                    'reg_lambda': cfg['model'].get('reg'), 'device': str(next(sup.dcrnn_model.parameters()).device)}
for split in ['val', 'test']:
    mis, width, mse, rmse, mae, out = sup.evaluate(split)
    n = len(np.load(os.path.join(a.data_dir, split + '.npz'))['y'])
    pred = np.stack(out['prediction']).astype(np.float32)   # (12, N_padded, nodes, 3) = lower, median, upper
    truth = np.stack(out['truth']).astype(np.float32)       # (12, N_padded, nodes)
    meta[f'{split}_samples'] = n
    meta[f'{split}_padded_samples'] = int(pred.shape[1])
    meta[f'{split}_authors_metrics'] = dict(mae=float(mae), mis=float(mis), width=float(width), mse=float(mse), rmse=float(rmse))
    result[f'{split}_pred'] = pred[:, :n]        # drop the copies of the last sample used to pad the final batch
    result[f'{split}_truth'] = truth[:, :n]
    print(f'{split}: authors-style (all 12 horizons, incl. padding)  MAE {mae:.4f}  MIS {mis:.4f}  '
          f'width {width:.4f}  MSE {mse:.4f}  RMSE {rmse:.4f}   [{n} samples, padded to {pred.shape[1]}]')
meta['seconds'] = round(time.time() - t0, 1)
os.makedirs(os.path.dirname(a.out), exist_ok=True)
np.savez_compressed(a.out, meta=json.dumps(meta), **result)
print('saved', a.out, f'({os.path.getsize(a.out)/1e6:.0f} MB, {meta["seconds"]} s)')

Writing /content/drive/MyDrive/CS637/p1/scripts/eval_checkpoint.py


In [ ]:
%%writefile {P1}/scripts/horizon_metrics.py
"""Per-horizon metrics from the prediction files written by eval_checkpoint.py.

Two readings of "15 / 30 / 60 min", because the paper does not say which it uses:
  step : the single forecast step at that lead time (step 3, 6, 12 of 12)
  avg  : the average over all steps up to that lead time (steps 1-3, 1-6, 1-12);
         avg at 60 min is the all-horizon average the authors' code prints
MIS is reported two ways:
  mis_authors : the code's mis_loss = (u - l) + |y - median| + 40*(y - u)+ + 40*(l - y)+
  mis_std     : the paper's Eq. 4 / standard interval score, without the |y - median| term
Readings where truth == 0 are excluded, as in the authors' masked losses.
"""
import numpy as np

LEADS = {15: 3, 30: 6, 60: 12}           # minutes -> number of 5-minute steps
RHO = 2.0 / 0.05                         # 2 / alpha for a 95% interval

def _metrics(pred, truth):
    m = truth != 0
    lo, mid, hi, y = pred[..., 0][m], pred[..., 1][m], pred[..., 2][m], truth[m]
    lo, mid, hi, y = (v.astype(np.float64) for v in (lo, mid, hi, y))
    err = mid - y
    pen = RHO * np.maximum(y - hi, 0) + RHO * np.maximum(lo - y, 0)
    mse = np.mean(err ** 2)
    return dict(mae=np.mean(np.abs(err)), mse=mse, rmse=np.sqrt(mse),
                mis_authors=np.mean(hi - lo + np.abs(err) + pen), mis_std=np.mean(hi - lo + pen),
                width=np.mean(hi - lo), coverage=np.mean((y >= lo) & (y <= hi)),
                crossed=np.mean((lo > mid) | (mid > hi)))

def horizon_table(pred, truth):
    """pred (12, N, nodes, 3), truth (12, N, nodes) -> list of dict rows."""
    rows = []
    for minutes, k in LEADS.items():
        rows.append(dict(view='step', minutes=minutes, **_metrics(pred[k - 1], truth[k - 1])))
        rows.append(dict(view='avg', minutes=minutes, **_metrics(pred[:k], truth[:k])))
    return rows

Writing /content/drive/MyDrive/CS637/p1/scripts/horizon_metrics.py


In [ ]:
%%writefile {P1}/scripts/check_sparse.py
# Optional check for the critique: does _build_sparse_matrix keep each value with its own (row, col)?
# Run from inside a variant folder.
import os, sys, numpy as np, torch
sys.path.insert(0, os.getcwd())
from lib import utils
from model.pytorch.dcrnn_cell import DCGRUCell
_, _, adj = utils.load_graph_data('data/sensor_graph/adj_mx.pkl')
for name, L in [('forward random walk', utils.calculate_random_walk_matrix(adj).T),
                ('backward random walk', utils.calculate_random_walk_matrix(adj.T).T)]:
    S = DCGRUCell._build_sparse_matrix(L).to_dense().cpu().numpy()
    D = L.toarray()
    print(f'{name}: max |torch matrix - intended matrix| = {np.abs(S - D).max():.4f} '
          f'(max entry {np.abs(D).max():.4f}); same sparsity pattern: {np.array_equal(S != 0, D != 0)}')

Writing /content/drive/MyDrive/CS637/p1/scripts/check_sparse.py


In [ ]:
if not all(os.path.exists(f'{DATA4}/{s}.npz') for s in ['train', 'val', 'test']):
    !python "{P1}/scripts/process_pems.py" --raw "{raw4}" --out "{DATA4}" --order 2,1,0
import math
n = {s: len(np.load(f'{DATA4}/{s}.npz')['y']) for s in ['train', 'val', 'test']}
print(n)
assert n == {'train': 11871, 'val': 1676, 'test': 3376}
# The authors' training logs show 940 batches after 20 epochs = 47 per epoch at batch size 256:
print('train batches per epoch:', math.ceil(n['train'] / 256), '(authors\' logs imply 47)')

train: x (11871, 12, 307, 3) y (11871, 12, 307, 3)
val: x (1676, 12, 307, 3) y (1676, 12, 307, 3)
test: x (3376, 12, 307, 3) y (3376, 12, 307, 3)
channel means [speed, occupancy, flow]: [6.34710e+01 5.30000e-02 2.11701e+02]
{'train': 11871, 'val': 1676, 'test': 3376}
train batches per epoch: 47 (authors' logs imply 47)


## 3. Smoke test: one checkpoint

We evaluate `gpde_quantile_phy_model_PEMSD4` at epoch 19, the checkpoint its config loads.
The authors' training log prints the test metrics for that epoch:
**MAE 2.0189 · MIS 37.8075 · MSE 19.8340 · RMSE 4.4535**.
If our authors-style numbers match to about 3 decimals, our data, code and checkpoint are the same as theirs. That is the go/no-go check for everything else.

In [ ]:
def evaluate(name, force=False):
    folder = MODELS[name]
    out = f'{OUT4}/{folder}.npz'
    if os.path.exists(out) and not force:
        print(f'{name}: already done -> {out}'); return out
    print(f'=== {name} ({folder})')
    r = subprocess.run([sys.executable, f'{P1}/scripts/eval_checkpoint.py', '--data_dir', DATA4, '--out', out,
                        '--log_dir', f'{P1}/logs/eval'], cwd=f'{AUTH}/{folder}', capture_output=True, text=True)
    print('\n'.join(l for l in r.stdout.splitlines() if 'authors-style' in l or l.startswith(('saved', 'NOTE'))))
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-5000:]); raise RuntimeError(f'{name} failed, paste the error above into Claude')
    return out

def meta(path):
    return json.loads(str(np.load(path)['meta']))

m = meta(evaluate('GPDE+Quantile+Phy'))['test_authors_metrics']
expected = dict(mae=2.0189, mis=37.8075, mse=19.8340, rmse=4.4535)
for k, v in expected.items():
    print(f'{k:5s} ours {m[k]:8.4f}   authors\' log {v:8.4f}   diff {m[k]-v:+.4f}')
ok = all(abs(m[k] - v) <= 0.01 * max(1, v / 10) for k, v in expected.items())
print('\nSMOKE TEST', 'PASSED: setup matches the authors' if ok else 'MISMATCH: paste this output into Claude before going on')

=== GPDE+Quantile+Phy (gpde_quantile_phy_model_PEMSD4)
val: authors-style (all 12 horizons, incl. padding)  MAE 2.2629  MIS 44.4474  width 4.6123  MSE 25.0999  RMSE 5.0100   [1676 samples, padded to 1792]
test: authors-style (all 12 horizons, incl. padding)  MAE 2.0194  MIS 37.8170  width 4.3887  MSE 19.8313  RMSE 4.4532   [3376 samples, padded to 3584]
saved /content/drive/MyDrive/CS637/outputs/pemsd4/gpde_quantile_phy_model_PEMSD4.npz (209 MB, 32.3 s)
mae   ours   2.0194   authors' log   2.0189   diff +0.0005
mis   ours  37.8170   authors' log  37.8075   diff +0.0095
mse   ours  19.8313   authors' log  19.8340   diff -0.0027
rmse  ours   4.4532   authors' log   4.4535   diff -0.0003

SMOKE TEST PASSED: setup matches the authors


## 4. Evaluate all six checkpoints

Finished models are skipped.

In [ ]:
for name in MODELS:
    evaluate(name)

=== Quantile (quantile_model_PEMSD4)
val: authors-style (all 12 horizons, incl. padding)  MAE 3.0472  MIS 68.4817  width 5.0257  MSE 40.2187  RMSE 6.3418   [1676 samples, padded to 1792]
test: authors-style (all 12 horizons, incl. padding)  MAE 2.6707  MIS 56.9723  width 4.8295  MSE 30.8541  RMSE 5.5546   [3376 samples, padded to 3584]
saved /content/drive/MyDrive/CS637/outputs/pemsd4/quantile_model_PEMSD4.npz (208 MB, 34.7 s)
=== GPDE+Quantile (gpde_quantile_model_PEMSD4)
val: authors-style (all 12 horizons, incl. padding)  MAE 2.3948  MIS 50.1806  width 4.4309  MSE 25.6795  RMSE 5.0675   [1676 samples, padded to 1792]
test: authors-style (all 12 horizons, incl. padding)  MAE 2.1362  MIS 42.6963  width 4.2032  MSE 20.4587  RMSE 4.5231   [3376 samples, padded to 3584]
saved /content/drive/MyDrive/CS637/outputs/pemsd4/gpde_quantile_model_PEMSD4.npz (209 MB, 37.4 s)
GPDE+Quantile+Phy: already done -> /content/drive/MyDrive/CS637/outputs/pemsd4/gpde_quantile_phy_model_PEMSD4.npz
=== MAEMI

## 5. Authors' training logs vs Table 1

Each variant folder ships the authors' `info.log` files, with test metrics after every epoch averaged over **all 12 horizons**.
For each model we take the test line at the epoch whose checkpoint the config loads, and put it next to Table 1's **"60 min"** row and our own evaluation.

Two claims get tested here: whether the "60 min" rows are really all-horizon averages, and whether the tables are means of five runs (Section 6.1.2) or a single run.

In [ ]:
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

# Table 1 of the paper, PEMSD4 columns (MAE, MIS, MSE, RMSE)
T1 = {
 'Quantile':          {15: (2.373, 42.771, 24.356, 3.465), 30: (2.419, 45.320, 25.316, 3.532), 60: (2.670, 56.975, 30.868, 3.889)},
 'GPDE+Quantile':     {15: (1.641, 25.043, 10.929, 2.472), 30: (1.722, 27.897, 12.321, 2.611), 60: (2.133, 42.622, 20.418, 3.281)},
 'GPDE+Quantile+Phy': {15: (1.539, 22.179, 10.231, 2.382), 30: (1.621, 24.744, 11.706, 2.523), 60: (2.018, 37.807, 19.834, 3.167)},
 'MAEMIS':            {15: (2.326, 39.423, 16.708, 3.217), 30: (2.382, 42.429, 18.002, 3.3148), 60: (3.167, 56.240, 25.347, 3.7920)},
 'GPDE+MAEMIS':       {15: (1.633, 24.601, 10.971, 2.493), 30: (1.708, 27.488, 12.392, 2.626), 60: (2.098, 43.089, 20.731, 3.267)},
 'GPDE+MAEMIS+Phy':   {15: (1.359, 16.962, 7.422, 2.039),  30: (1.623, 24.807, 11.652, 2.513), 60: (2.044, 38.501, 20.882, 3.244)},
}
pat = re.compile(r'Epoch \[(\d+)/\d+\].*test_mae:([\d.]+),test_mis: ([\d.]+), test_width: ([\d.]+), test_mse: ([\d.]+), test_rmse: ([\d.]+)')
rows = []
for name, folder in MODELS.items():
    mt = meta(f'{OUT4}/{folder}.npz'); ep = mt['epoch']
    n_runs, hit = 0, None
    for log in sorted(glob.glob(f'{AUTH}/{folder}/data/model/dcrnn_*/info.log')):
        lines = [pat.search(l) for l in open(log)]
        lines = [l for l in lines if l]
        if lines: n_runs += 1
        txt = open(log).read()
        if f'saving to models/epo{ep}.tar' in txt:
            hit = [l for l in lines if int(l.group(1)) == ep][-1]
    t = T1[name][60]; ours = mt['test_authors_metrics']
    rows.append(dict(model=name, epoch=ep, logged_runs=n_runs,
        T1_60_mae=t[0], log_mae=float(hit.group(2)), our_mae=ours['mae'],
        T1_60_mis=t[1], log_mis=float(hit.group(3)), our_mis=ours['mis'],
        T1_60_mse=t[2], log_mse=float(hit.group(5)), our_mse=ours['mse'],
        T1_60_rmse=t[3], log_rmse=float(hit.group(6)), our_rmse=ours['rmse']))
logs = pd.DataFrame(rows).round(4)
logs.to_csv(f'{P1}/results/pemsd4_table1_vs_authors_logs.csv', index=False)
display(logs)

,model,epoch,logged_runs,T1_60_mae,log_mae,our_mae,T1_60_mis,log_mis,our_mis,T1_60_mse,log_mse,our_mse,T1_60_rmse,log_rmse,our_rmse
0,Quantile,18,1,2.670,2.6702,2.6707,56.975,56.9755,56.9723,30.868,30.8685,30.8541,3.889,5.5559,5.5546
1,GPDE+Quantile,18,1,2.133,2.1338,2.1362,42.622,42.6230,42.6963,20.418,20.4184,20.4587,3.281,4.5187,4.5231
2,GPDE+Quantile+Phy,19,1,2.018,2.0189,2.0194,37.807,37.8075,37.8170,19.834,19.8340,19.8313,3.167,4.4535,4.4532
3,MAEMIS,18,1,3.167,2.6805,2.6831,56.240,56.2402,56.2917,25.347,25.3472,25.3483,3.792,5.0346,5.0347
4,GPDE+MAEMIS,16,1,2.098,2.0986,2.1010,43.089,43.0895,43.1232,20.731,20.7315,20.7783,3.267,4.5532,4.5583
5,GPDE+MAEMIS+Phy,19,4,2.044,2.0448,2.0461,38.501,38.5015,38.5349,20.882,20.8828,20.8989,3.244,4.5698,4.5715


## 6. Our per-horizon metrics vs Table 1

The paper doesn't say whether "15 min" means the single 15-minute step or the average over the first 15 minutes, so we compute both:
- `step`: forecast step 3, 6 or 12
- `avg`: steps 1–3, 1–6 or 1–12

We also report MIS two ways. `mis_authors` is what the code computes, and it includes a |y − median| term. `mis_std` is the paper's Eq. 4. Coverage is the share of true speeds inside the 95% interval; the paper never reports it.

In [ ]:
sys.path.insert(0, f'{P1}/scripts')
import importlib, horizon_metrics; importlib.reload(horizon_metrics)
rows = []
for name, folder in MODELS.items():
    z = np.load(f'{OUT4}/{folder}.npz')
    for r in horizon_metrics.horizon_table(z['test_pred'], z['test_truth']):
        t = T1[name][r['minutes']]
        r.update(model=name, T1_mae=t[0], T1_mis=t[1], T1_mse=t[2], T1_rmse=t[3])
        for k, tk in [('mae', 'T1_mae'), ('mis_authors', 'T1_mis'), ('mse', 'T1_mse')]:
            r[f'{k}_diff_%'] = 100 * (r[k] - r[tk]) / r[tk]
        rows.append(r)
H = pd.DataFrame(rows)[['model', 'view', 'minutes', 'mae', 'T1_mae', 'mae_diff_%', 'mis_authors', 'T1_mis', 'mis_authors_diff_%',
                         'mis_std', 'mse', 'T1_mse', 'mse_diff_%', 'rmse', 'T1_rmse', 'width', 'coverage', 'crossed']].round(4)
H.to_csv(f'{P1}/results/pemsd4_horizons_vs_table1.csv', index=False)

print('Which reading of 15/30/60 min matches Table 1? Mean |% diff| in MAE, MIS, MSE:')
H['abs_diff'] = H[['mae_diff_%', 'mis_authors_diff_%', 'mse_diff_%']].abs().mean(axis=1)
display(H.pivot_table(index=['model'], columns=['view', 'minutes'], values='abs_diff').round(2))
display(H.drop(columns='abs_diff'))

Which reading of 15/30/60 min matches Table 1? Mean |% diff| in MAE, MIS, MSE:


view                 avg               step              
minutes               15    30    60     15     30     60
model                                                    
GPDE+MAEMIS        18.35  2.85  3.39   3.56  48.85  61.23
GPDE+MAEMIS+Phy     2.16  2.96  3.64  32.71  51.26  66.95
GPDE+Quantile      18.57  3.07  3.69   4.35  49.11  60.51
GPDE+Quantile+Phy  21.18  2.31  2.93   3.64  51.12  59.26
MAEMIS             10.73  3.37  7.13   6.63  32.57  34.84
Quantile            6.67  2.93  3.09   6.43  22.49  31.70

,model,view,minutes,mae,T1_mae,mae_diff_%,mis_authors,T1_mis,mis_authors_diff_%,mis_std,mse,T1_mse,mse_diff_%,rmse,T1_rmse,width,coverage,crossed
0,Quantile,step,15,2.4280,2.373,2.3178,47.7646,42.771,11.6752,45.3366,25.6468,24.356,5.2996,5.0643,3.4650,4.5627,0.6961,0.0
1,Quantile,avg,15,2.3188,2.373,-2.2854,36.9958,42.771,-13.5025,34.6771,23.3294,24.356,-4.2149,4.8301,3.4650,7.3124,0.7993,0.0
2,Quantile,step,30,2.6952,2.419,11.4159,59.6239,45.320,31.5621,56.9288,31.5140,25.316,24.4824,5.6137,3.5320,4.1474,0.6459,0.0
3,Quantile,avg,30,2.4633,2.419,1.8313,46.4569,45.320,2.5085,43.9936,26.4421,25.316,4.4481,5.1422,3.5320,5.7893,0.7308,0.0
4,Quantile,step,60,3.1758,2.670,18.9453,78.3507,56.975,37.5176,75.1748,42.7947,30.868,38.6376,6.5418,3.8890,3.8280,0.5720,0.0
5,Quantile,avg,60,2.7215,2.670,1.9294,58.5460,56.975,2.7574,55.8245,32.2870,30.868,4.5969,5.6822,3.8890,4.8730,0.6674,0.0
6,GPDE+Quantile,step,15,1.7113,1.641,4.2810,26.1354,25.043,4.3620,24.4241,11.4098,10.929,4.3996,3.3778,2.4720,4.6929,0.7842,0.0
7,GPDE+Quantile,avg,15,1.4822,1.641,-9.6744,19.3140,25.043,-22.8766,17.8318,8.3970,10.929,-23.1673,2.8978,2.4720,5.5244,0.8555,0.0
8,GPDE+Quantile,step,30,2.1818,1.722,26.7041,43.6179,27.897,56.3534,41.4361,20.2409,12.321,64.2793,4.4990,2.6110,3.9262,0.6456,0.0
9,GPDE+Quantile,avg,30,1.7579,1.722,2.0848,28.6674,27.897,2.7616,26.9095,12.8584,12.321,4.3617,3.5859,2.6110,4.8232,0.7710,0.0


In [ ]:
print('Interval coverage on the test set (target 0.95)')
display(H[H.view == 'step'].pivot_table(index='model', columns='minutes', values='coverage').round(3))
print('\nGain from the physics loss at each horizon (step view, MAE): GPDE+X vs GPDE+X+Phy')
s = H[H.view == 'step'].set_index(['model', 'minutes'])['mae']
for base in ['Quantile', 'MAEMIS']:
    for mnt in [15, 30, 60]:
        a, b = s[(f'GPDE+{base}', mnt)], s[(f'GPDE+{base}+Phy', mnt)]
        print(f'  {base:8s} {mnt:2d} min: {a:.3f} -> {b:.3f}  ({100*(b-a)/a:+.1f}%)')

Interval coverage on the test set (target 0.95)


minutes,15,30,60
model,,,
GPDE+MAEMIS,0.757,0.633,0.501
GPDE+MAEMIS+Phy,0.840,0.737,0.639
GPDE+Quantile,0.784,0.646,0.518
GPDE+Quantile+Phy,0.837,0.743,0.653
MAEMIS,0.590,0.516,0.450
Quantile,0.696,0.646,0.572



Gain from the physics loss at each horizon (step view, MAE): GPDE+X vs GPDE+X+Phy
  Quantile 15 min: 1.711 -> 1.598  (-6.6%)
  Quantile 30 min: 2.182 -> 2.062  (-5.5%)
  Quantile 60 min: 2.919 -> 2.731  (-6.4%)
  MAEMIS   15 min: 1.680 -> 1.609  (-4.2%)
  MAEMIS   30 min: 2.131 -> 2.075  (-2.6%)
  MAEMIS   60 min: 2.836 -> 2.849  (+0.5%)


In [ ]:
readme = '''PEMSD4 predictions from the authors' shipped checkpoints (Person 1, notebook 1).
One file per model: <authors' folder name>.npz

Keys
  val_pred, test_pred    float32 (12, N, 307, 3)   last axis = [lower 2.5%, median, upper 97.5%], speed in mph
  val_truth, test_truth  float32 (12, N, 307)      true speed in mph
  meta                   JSON string: variant, epoch, lambda, sample counts, authors-style metrics
Axis 0 = forecast step 1..12 (5, 10, ..., 60 min ahead). N = 1676 (val), 3376 (test).
The authors' evaluation pads the last batch with copies of the final sample; those copies are removed here.
Mask truth == 0 when computing metrics (the authors do; PEMSD4 speed has no zeros, so it changes nothing here).

Load:  z = np.load(path); pred = z['test_pred']; meta = json.loads(str(z['meta']))
'''
open(f'{OUT4}/README.txt', 'w').write(readme)
for f in sorted(glob.glob(f'{OUT4}/*.npz')):
    print(f'{os.path.basename(f):45s} {os.path.getsize(f)/1e6:6.0f} MB')

gpde_maemis_model_PEMSD4.npz                     208 MB
gpde_maemis_phy_model_PEMSD4_0.005.npz           210 MB
gpde_quantile_model_PEMSD4.npz                   209 MB
gpde_quantile_phy_model_PEMSD4.npz               209 MB
maemis_model_PEMSD4.npz                          209 MB
quantile_model_PEMSD4.npz                        208 MB


## 7. Sparse graph matrix

`DCGRUCell._build_sparse_matrix` (inherited from DCRNN-PyTorch) re-sorts the sparse **indices** with `np.lexsort` but passes the **values** in their old order.
If that's a real bug, every diffusion convolution uses a scrambled graph matrix. This cell checks it on the PEMSD4 graph in a few seconds.

In [ ]:
!cd "{AUTH}/gpde_quantile_phy_model_PEMSD4" && python "{P1}/scripts/check_sparse.py"

I0000 00:00:1792562921.859711    3367 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792562926.281942    3367 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/content/drive/MyDrive/CS637/uq4phy_traffic_network/gpde_quantile_phy_model_PEMSD4/model/pytorch/dcrnn_cell.py:84: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  L = torch.sparse_coo_tensor(indices.T,L.data, L.shape, devi